In [ ]:
# !pip install "sahi[ultralytics]"

In [ ]:
# !pip install "numpy<2.0.0,>=1.26.0" --force-reinstall

In [ ]:
# !pip install "numpy<2.0.0,>=1.26.0" "opencv-python<4.10.0"

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
import mediapipe as mp
import torch
import sahi
import ultralytics

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe:", mp.__version__)
print("PyTorch:", torch.__version__)
print("SAHI:", sahi.__version__)
print("Ultralytics:", ultralytics.__version__)

In [ ]:
%pip check

In [ ]:
%pip install --force-reinstall "numpy==1.26.4" "opencv-python==4.9.0.80"

#### PATHS

In [ ]:
from pathlib import Path
import cv2
import os

# Original dataset
DATASET = Path("/home/vedant/MINOR PROJECT/PPE KIT/20250731-ppe2286y")

# New sliced dataset
OUTPUT = Path("/home/vedant/MINOR PROJECT/PPE KIT/sliced dataset")

# Slicing configuration
SLICE_SIZE = 512
OVERLAP = 0.20

print("Paths configured!")

### CREATE SLICING FUNCTION

In [ ]:
def get_positions(length, slice_size, overlap):

    if length <= slice_size:
        return [0]

    step = int(slice_size * (1 - overlap))

    positions = list(range(0, length - slice_size + 1, step))

    last = length - slice_size

    if positions[-1] != last:
        positions.append(last)

    return positions

In [ ]:
def slice_dataset_image(image_path, label_path, img_output, lbl_output):

    image = cv2.imread(str(image_path))

    if image is None:
        return 0

    H, W = image.shape[:2]

    boxes = []

    if label_path.exists():

        with open(label_path) as f:

            for line in f:

                values = line.strip().split()

                if len(values) != 5:
                    continue

                cls, xc, yc, bw, bh = map(float, values)

                x1 = (xc - bw / 2) * W
                y1 = (yc - bh / 2) * H
                x2 = (xc + bw / 2) * W
                y2 = (yc + bh / 2) * H

                boxes.append((int(cls), x1, y1, x2, y2))

    xs = get_positions(W, SLICE_SIZE, OVERLAP)
    ys = get_positions(H, SLICE_SIZE, OVERLAP)

    count = 0

    for y in ys:

        for x in xs:

            xe = min(x + SLICE_SIZE, W)
            ye = min(y + SLICE_SIZE, H)

            crop = image[y:ye, x:xe]

            ch, cw = crop.shape[:2]

            new_labels = []

            for cls, bx1, by1, bx2, by2 in boxes:

                original_area = (bx2 - bx1) * (by2 - by1)

                if original_area <= 0:
                    continue

                # Clip bounding box to crop boundaries
                nx1 = max(bx1, x)
                ny1 = max(by1, y)
                nx2 = min(bx2, xe)
                ny2 = min(by2, ye)

                if nx2 <= nx1 or ny2 <= ny1:
                    continue

                intersection = (nx2 - nx1) * (ny2 - ny1)

                visibility = intersection / original_area

                # Ignore heavily cropped objects
                if visibility < 0.30:
                    continue

                # Coordinates relative to cropped image
                nx1 -= x
                nx2 -= x
                ny1 -= y
                ny2 -= y

                # Normalize YOLO coordinates
                xc = ((nx1 + nx2) / 2) / cw
                yc = ((ny1 + ny2) / 2) / ch

                bw = (nx2 - nx1) / cw
                bh = (ny2 - ny1) / ch

                new_labels.append(
                    f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
                )

            filename = f"{image_path.stem}_{x}_{y}"

            cv2.imwrite(
                str(img_output / f"{filename}.jpg"),
                crop
            )

            with open(lbl_output / f"{filename}.txt", "w") as f:
                f.write("\n".join(new_labels))

            count += 1

    return count

#### RUN SLICING ON COMLETE DATSET

In [ ]:
from pathlib import Path

SPLIT_MAP = {
    "train": "train",
    "val": "valid"
}

for split, original_split in SPLIT_MAP.items():

    image_dir = DATASET / original_split / "images"
    label_dir = DATASET / original_split / "labels"

    img_output = OUTPUT / "images" / split
    lbl_output = OUTPUT / "labels" / split

    img_output.mkdir(parents=True, exist_ok=True)
    lbl_output.mkdir(parents=True, exist_ok=True)

    image_paths = []

    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
        image_paths.extend(image_dir.glob(ext))

    total = 0

    print(f"\nSlicing {split} images...")

    for image_path in image_paths:

        label_path = label_dir / (image_path.stem + ".txt")

        total += slice_dataset_image(
            image_path,
            label_path,
            img_output,
            lbl_output
        )

    print("Original images:", len(image_paths))
    print("Generated slices:", total)

print("\nDATASET SLICING COMPLETED!")

In [ ]:
import yaml

config = {
    "path": str(OUTPUT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": 4,
    "names": [
        "Helmet",
        "No Helmet",
        "No Safety Vest",
        "Safety Vest"
    ]
}

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("data.yaml created!")

#### fine tune model

In [ ]:
from ultralytics import YOLO

model = YOLO("/home/vedant/MINOR PROJECT/PPE KIT/runs/detect/train-28/weights/best.pt")

results = model.train(
    data=str(OUTPUT / "data.yaml"),
    epochs=35,
    imgsz=800,
    batch=4,
    device=0,
    workers=2,
    lr0=0.0001,
    patience=8,
    optimizer="AdamW",
    plots=True,
    save=True

)

In [ ]:
# import numpy as np
# import torch
# import cv2

# print("NumPy:", np.__version__)
# print("NumPy location:", np.__file__)

# print("PyTorch:", torch.__version__)
# print("OpenCV:", cv2.__version__)

# # Test NumPy-PyTorch compatibility
# x = np.array([1, 2, 3], dtype=np.float32)

# y = torch.from_numpy(x)

# print("Compatibility test:", y)

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# Load your trained PPE model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="/home/vedant/MINOR PROJECT/PPE KIT/runs/detect/train-4/weights/best.pt",
    confidence_threshold=0.3,
    device="cuda:0"
)

# Run SAHI sliced inference
result = get_sliced_prediction(
    "/home/vedant/MINOR PROJECT/PPE KIT/20250731-ppe2286y/train/images/00005_jpg.rf.3178537f5af58d6bc65137481af98e7a.jpg",
    detection_model,
    slice_height=480,
    slice_width=480,
    overlap_height_ratio=0.20,
    overlap_width_ratio=0.20
)

# Save detection result
result.export_visuals(
    export_dir="/home/vedant/MINOR PROJECT/PPE KIT/sliced dataset",
    file_name="ppe_detection3"
)

# Print detections
for prediction in result.object_prediction_list:
    print(
        prediction.category.name,
        prediction.score.value,
        prediction.bbox.to_xyxy()
    )

### FOR VIDEO

In [ ]:
import cv2
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions

# 1. Initialize SAHI detection model once
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="/home/vedant/MINOR PROJECT/PPE KIT/runs/detect/train-4/weights/best.pt",
    confidence_threshold=0.35,
    device="cuda:0"
)

# 2. Open input video file
input_path = ""
cap = cv2.VideoCapture(input_path)

# Extract video properties for the writer
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 3. Setup VideoWriter (saves as MP4)
output_path = "/home/vedant/MINOR PROJECT/PPE KIT/annotated_output 2.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# # Optional: Set frame skip if you want to speed up processing, otherwise process every frame (frame_skip = 1)
# frame_skip = 1 
# frame_count = 0

# print(f"Processing and saving video to: {output_path}...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_skip > 1 and frame_count % frame_skip != 0:
        continue

    # 4. Convert frame from BGR to RGB for SAHI
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 5. Perform sliced inference
    result = get_sliced_prediction(
        frame_rgb,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2
        
        
    #     postprocess_match_threshold=0.30        
    )

    # 6. Render clean annotations using SAHI visualizer
    annotated_dict = visualize_object_predictions(
        image=frame_rgb,
        object_prediction_list=result.object_prediction_list,
        rect_th=1,
        text_th=1,
        text_size=0.4
    )

    # 7. Convert back to BGR for OpenCV
    annotated_frame = cv2.cvtColor(annotated_dict["image"], cv2.COLOR_RGB2BGR)

    # 8. Write the annotated frame to output file
    out.write(annotated_frame)

    # Optional live display preview
    cv2.imshow("Processing CCTV Stream", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release all resources
cap.release()
out.release()
cv2.destroyAllWindows()

print("Video saving complete!")

Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing predictio